# EM Tutorial: Two-State Sensorimotor Adaptation

This notebook demonstrates how to use the `albert_em` Python package to: 
1. Define an experimental paradigm containing perturbation and error-clamp trials.
2. Simulate noisy behavior from a known two-state model.
3. Fit the model parameters with the Expectation-Maximization (EM) algorithm.
4. Visualize behavioral data, hidden states, and convergence (log-likelihood).

The implementation uses Numba for performance and falls back to pure Python if disabled.

In [ ]:
# Imports and environment setup
import os, sys, time, math
from pathlib import Path
# Ensure src/ is on the path if working from a clone without install
repo_root = Path('..').resolve()
src_path = repo_root / 'src'
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import numpy as np
import matplotlib.pyplot as plt
from albert_em import (
    generalized_expectation_maximization,
    two_state_simulation_with_noise,
    two_state_simulation_without_noise
)

# Detect whether numba optimization is active (best-effort)
try:
    import numba
    USING_NUMBA = (os.getenv('ALBERT_EM_DISABLE_NUMBA', '0') != '1')
except ImportError:
    USING_NUMBA = False

print(f'Numba acceleration active: {USING_NUMBA}')

: 

In [ ]:
# Part 1: Define paradigm (perturbations + error-clamp trials)
r = np.concatenate([
    np.zeros(20),           # baseline
    30 * np.ones(50),       # adaptation perturbation (30)
    np.full(20, np.nan),    # error-clamp trials
    np.zeros(30)            # washout
])
EC = np.concatenate([
    np.zeros(70),           # normal trials
    np.ones(20),            # error-clamp
    np.zeros(30)            # normal
])
EC_value = np.concatenate([
    np.full(70, np.nan),    # not clamped
    np.zeros(20),           # clamped error value = 0
    np.full(30, np.nan)     # not clamped
])
c = np.array([1.0, 1.0])  # equal weighting of states
N = len(r)
print(f'Total trials: {N}')

In [ ]:
# Part 2: Simulate noisy behavior from known parameters
true_params = np.array([
    0.985,  # aS slow retention
    0.556,  # aF fast retention
    0.097,  # bS slow error sensitivity
    0.213,  # bF fast error sensitivity
    0.0,    # xS1 initial slow state
    0.0,    # xF1 initial fast state
    1.694,  # sigmax2 state update variance
    1.037,  # sigmau2 motor variance
    0.0     # sigma12 initial state variance
])
np.random.seed(5)
y_obs, xS_true, xF_true = two_state_simulation_with_noise(true_params, r, EC, EC_value, c)
print('Simulated motor output (first 10):', y_obs[:10])

In [ ]:
# Part 3: Configure EM algorithm
search_space = np.array([
    [0, 1.1], [0, 1.1], [0, 1], [0, 1],
    [-30, 30], [-30, 30],
    [1e-7, 10], [1e-7, 10], [1e-7, 10]
])
constraints = np.array([0.001, 0.001])  # enforce aS>aF+deltaA and bF>bS+deltaB
initial_guess = np.array([0.95, 0.7, 0.05, 0.30, 0, 0, 2, 2, 5])
num_iterations = 60  # fewer iterations for notebook demo
start = time.time()
fitted_params, likelihoods = generalized_expectation_maximization(
    initial_guess, y_obs, r, EC, EC_value, c, search_space, constraints, num_iterations
)
elapsed = time.time() - start
print(f'EM finished in {elapsed:.2f}s')
print('Initial likelihood:', likelihoods[0])
print('Final likelihood:', likelihoods[-1])
improvement = likelihoods[-1] - likelihoods[0]
print(f'Improvement: {improvement:.2f}')
# Basic monotonicity check (allow small numerical noise)
non_decreasing = np.all(np.diff(likelihoods) > -1e-6)
print('Likelihood non-decreasing:', non_decreasing)

In [ ]:
# Part 4: Inspect fitted parameters vs truth
names = ['aS','aF','bS','bF','xS1','xF1','sigmax2','sigmau2','sigma12']
print(f'\nParameter Comparison:')
print(f'{
:<8} {
:>10} {
:>10} {
:>10}')
for i, n in enumerate(names):
    print(f'{n:<8} {true_params[i]:>10.4f} {initial_guess[i]:>10.4f} {fitted_params[i]:>10.4f}')

In [ ]:
# Part 5: Simulate deterministic (noise-free) trajectories with fitted params
y_fit, xS_fit, xF_fit = two_state_simulation_without_noise(fitted_params, r, EC, EC_value, c)
# Plot behavior and state trajectories
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
axes[0].plot(y_obs, 'k', label='Observed noisy')
axes[0].plot(y_fit, '--r', label='EM fit (deterministic)')
axes[0].plot(np.nan_to_num(r, nan=0.0), ':b', label='Perturbation')
axes[0].set_ylabel('Motor Output')
axes[0].legend(loc='upper right')
axes[1].plot(xS_true, 'k', label='True Slow (noisy)')
axes[1].plot(xS_fit, '--r', label='Fitted Slow')
axes[1].set_ylabel('Slow State')
axes[1].legend(loc='upper right')
axes[2].plot(xF_true, 'k', label='True Fast (noisy)')
axes[2].plot(xF_fit, '--r', label='Fitted Fast')
axes[2].set_ylabel('Fast State')
axes[2].set_xlabel('Trial')
axes[2].legend(loc='upper right')
fig.suptitle('Behavior and Hidden States: True vs EM Fit')
plt.tight_layout()
plt.show()

In [ ]:
# Part 6: Plot log-likelihood convergence
plt.figure(figsize=(8,4))
plt.plot(likelihoods, 'k')
plt.xlabel('EM Iteration')
plt.ylabel('log L(y|θ)')
plt.title('Incomplete Log-Likelihood Convergence')
plt.grid(alpha=0.3)
plt.show()

## Next Steps
- Adjust `num_iterations` for higher accuracy.
- Provide real experimental data in place of simulated `y_obs`.
- Enable/disable Numba via environment variable: `ALBERT_EM_DISABLE_NUMBA=1`.
- Add batch fitting for multiple subjects.

This concludes the tutorial notebook.